# Training the hourly flood model with PyTorch Lightning

## Purpose

This notebook trains, step by step, the kind of model caudal issues every hour at the Russian River flood gauges: Hopland (HOPC1), Healdsburg (HEAC1) and Guerneville (GUEC1), with leads of 1 to 120 hours. It uses caudal's own pieces for the data, the windows and the network, and wraps them in a PyTorch Lightning module, so the training can be watched over epochs with Lightning's `Trainer`, its callbacks and its loggers.

| Section | What it shows |
|---|---|
| 1. Settings | How long to train, and the switch for a full run |
| 2. The data | One hourly frame per gauge: weather, rain totals, the gauge's flow and the flows upstream |
| 3. Windows, masking and the anchor | How one training sample is built, and how one model learns every lead |
| 4. The network | `QuantileLSTM` and its monotone quantile head |
| 5. The loss | The pinball loss, weighted by flow |
| 6. The Lightning module | `training_step`, `validation_step` and the optimiser |
| 7. Training | `Trainer` with checkpointing, early stopping, learning-rate monitoring and two loggers |
| 8. Training curves | Train and validation loss by epoch |
| 9. Scores | CRPS against persistence on a held-out year, by lead |
| 10. A storm | The forecast issued before the January 2017 flood, against what happened |

The issued product, `quantile_flow_hourly_v2_s0`, is the average of three such models that differ in their inputs and loss (caudal #28). This notebook trains one of them, `weighted_log`, and the switch in section 1 trains `full` instead.

**Before running:** use the `embalse` kernel (caudal's conda environment). The hourly archives must exist under `data/`: `python -m caudal.sources.usgs 11462500 11464000 11467000 11465350` and `python -m caudal.sources.openmeteo`. Install the notebook extra: `pip install -e ".[notebook]"`.

## 1. Settings

The defaults train the issued model's network on a sample of windows for 10 epochs, about a minute on an Apple GPU, so the whole notebook runs in a few minutes. A quick run is under-trained: its scores in section 9 are below the issued model's. `FULL_RUN = True` uses the issued model's size and length.

| Setting | Quick | Full run | What it controls |
|---|---|---|---|
| `HIDDEN` | 128 | 128 | LSTM hidden units |
| `EPOCHS` | 10 | 20 | Passes of the training loop |
| `SAMPLES` | 100,000 | 250,000 | Windows drawn at random per epoch |
| `CONFIG` | `weighted_log` | `weighted_log` | What the model reads: `v1`, `weighted_log` or `full` |

In [ ]:
FULL_RUN = False
CONFIG = "weighted_log"          # "v1", "weighted_log" or "full"
HIDDEN, EPOCHS, SAMPLES = (128, 20, 250_000) if FULL_RUN else (128, 10, 100_000)
FIT_END = "2015-09-30 23:00"                                # training: water years 1988 to 2015
VAL_START, VAL_END = "2016-10-01 00:00", "2017-09-30 23:00"  # validation: water year 2017, with the January floods
SEED = 0

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from torch.utils.data import DataLoader, RandomSampler

from caudal import flow_hourly as H
from caudal.config import RESULTS_DIR
from caudal.models.quantile_lstm import DEFAULT_QUANTILES, QuantileLSTM, pinball_loss
from caudal.models.sequence import SequenceDataset
from caudal.points import GAUGES
from caudal.scores import crps_ensemble
from caudal.training import Anchor, MaskRecent

L.seed_everything(SEED)
H.configure(H.CONFIGS[CONFIG])
H.set_years(FIT_END, VAL_START, VAL_END)
print(f"config {CONFIG}: {len(H.FEATURES)} inputs per hour, lookback {H.LOOKBACK} h, leads 1 to {H.LEADS} h")

## 2. The data

### Overview

Each gauge gets one hourly frame. A row holds, for hour $t$:

- the basin mean of the weather of hour $t + 1$ (rain, temperature, and for `full` also soil moisture and evaporation), from the rainfall points inside the basin outline;
- running rainfall totals over the last 24 hours to 30 days (90 days for `full`), and the season as $\sin$ and $\cos$ of the day of year;
- the observed flow of hour $t$ at the gauge, and at the points upstream (Hopland for Healdsburg; Healdsburg and Dry Creek for Guerneville; the dam releases for `full`).

The weather sits one hour ahead of the flow on purpose: the window that ends just before a target hour then holds the rain up to that hour and the flow up to the hour before.

`build_regional` builds the three frames, scales the weather by statistics pooled over the fit years, and scales each gauge's flow by its own spread, so the three gauges share one model.

In [ ]:
t0 = time.time()
frames = H.build_regional(list(GAUGES))
print(f"built in {time.time() - t0:.0f} s")
for lid, fr in frames["raw"].items():
    print(f"{lid}: {len(fr):,} hours, {fr.index[0]:%Y-%m-%d} to {fr.index[-1]:%Y-%m-%d}, columns {list(fr.columns)[:6]} ...")
frames["raw"]["GUEC1"].loc["2017-01-08":"2017-01-08 05:00", ["precip", "precip_24h", "flow", "up1", "up2"]]

In [ ]:
fr = frames["raw"]["GUEC1"].loc["2017-01-03":"2017-01-15"]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(fr.index, fr["flow"], color="k", lw=1.5, label="Guerneville (Hacienda Bridge)")
ax.plot(fr.index, fr["up1"], lw=1, label="Healdsburg, upstream")
ax.set_ylabel("flow, cfs"); ax.legend(loc="upper left")
ax2 = ax.twinx(); ax2.bar(fr.index, fr["precip"], width=1 / 24, color="tab:blue", alpha=0.35); ax2.set_ylabel("basin rain, mm/h")
ax.set_title("January 2017 at Guerneville: rain, the upstream gauge, and the river")
plt.show()

## 3. Windows, masking and the anchor

### One sample

`SequenceDataset` cuts each frame into windows of `LOOKBACK` = 240 hours. The target is the flow of the hour right after the window. Windows never cross a gap in the record.

### One model for every lead

At an issue, the model knows the flow only up to the issue hour, and the forecast for lead $L$ is $L$ hours later. In training, `MaskRecent` hides the newest $m$ hours of every flow input, with $m$ drawn at random from 0 to 126: their values become 0 and their missing flags 1. The same network then learns lead 1 when $m = 0$ and lead 120 when $m = 119$.

### The anchor

`Anchor` turns the target into its change from the newest flow the window still shows. A model that has learned nothing is then persistence, and every forecast starts from the river.

### The weight

For `weighted_log` and `full`, flow is modelled as $\log(1 + q)$, so a recession is a proportional fall and no forecast can reach zero. Each window also carries a weight, the target hour's flow over the gauge's mean flow, so a flood hour counts in the loss about as it would in cfs.

In [ ]:
weight = "weight" if H.WEIGHTED else None
train_windows = SequenceDataset(list(frames["train"].values()), H.FEATURES, H.TARGET, H.LOOKBACK, weight)
train_ds = Anchor(MaskRecent(train_windows, H.FEATURES, H.MAX_HIDDEN, random=True, pairs=H.PAIRS), H.FEATURES, H.PAIRS[0])
print(f"{len(train_windows):,} training windows over {len(frames['train'])} gauges")

x, y, *w = train_ds[123_456]
flow, flag = (H.FEATURES.index(c) for c in H.PAIRS[0])
hidden_hours = int(x[:, flag].sum())
print(f"one window: {tuple(x.shape)} (hours x inputs); the newest {hidden_hours} hours of flow are hidden")
print(f"target: change of {float(y):+.3f} (scaled) from the newest visible flow; weight {float(w[0]) if w else 1:.2f}")

In [ ]:
val_frames = {lid: H.prep(fr.loc[VAL_START:VAL_END], frames["f_mean"], frames["f_std"], frames["scale"][lid])
              for lid, fr in frames["raw"].items()}
val_windows = SequenceDataset(list(val_frames.values()), H.FEATURES, H.TARGET, H.LOOKBACK, weight)
val_ds = Anchor(MaskRecent(val_windows, H.FEATURES, 24, pairs=H.PAIRS), H.FEATURES, H.PAIRS[0])  # a fixed 24-hour lead
train_loader = DataLoader(train_ds, batch_size=H.BATCH_SIZE, sampler=RandomSampler(train_ds, replacement=True, num_samples=SAMPLES))
val_loader = DataLoader(val_ds, batch_size=2048, sampler=RandomSampler(val_ds, replacement=False, num_samples=min(20_000, len(val_ds)),
                                                                        generator=torch.Generator().manual_seed(SEED)))
print(f"{len(val_windows):,} validation windows in water year 2017; each epoch reads {SAMPLES:,} training and 20,000 validation windows")

## 4. The network

### Overview

`QuantileLSTM` reads the 240-hour window with one LSTM layer and maps its last hidden state to seven quantiles: 5, 25, 50, 75, 90, 95 and 99%. The head is monotone by construction: the first output is free, and each next quantile adds a softplus step, so quantiles can never cross:

$$\hat q_1 = z_1, \qquad \hat q_k = \hat q_{k-1} + \operatorname{softplus}(z_k), \quad k = 2, \dots, 7.$$

In [ ]:
net = QuantileLSTM(n_features=len(H.FEATURES), hidden_size=HIDDEN)
print(net)
print(f"{sum(p.numel() for p in net.parameters()):,} parameters; quantiles {net.quantiles}")

## 5. The loss

For quantile level $\tau$, target $y$ and prediction $\hat q_\tau$, the pinball loss is

$$\rho_\tau(y, \hat q_\tau) = \max\big(\tau\,(y - \hat q_\tau),\; (\tau - 1)\,(y - \hat q_\tau)\big).$$

It is minimised by the true $\tau$ quantile, so a model trained on it learns a distribution, not a single value. Averaged over the seven levels and weighted over a batch of $N$ windows,

$$\mathcal{L} = \frac{\sum_{i=1}^{N} w_i \,\tfrac{1}{7}\sum_{\tau} \rho_\tau(y_i, \hat q_{\tau,i})}{\sum_{i=1}^{N} w_i},$$

with $w_i$ the weight of section 3 (all ones for `v1`). This is `caudal.models.quantile_lstm.pinball_loss`.

## 6. The Lightning module

### Overview

A `LightningModule` holds the network and says what one step does; Lightning runs the loop: devices, gradient clipping, logging, checkpoints. The module below adds nothing to the model. It calls caudal's `QuantileLSTM` and `pinball_loss` as they are.

| Hook | What it does here |
|---|---|
| `training_step` | Pinball loss of a batch, logged as `train_pinball` per step and per epoch |
| `validation_step` | The same on the 24-hour validation windows, logged as `val_pinball` |
| `configure_optimizers` | Adam, and a scheduler that halves the learning rate when `val_pinball` stops improving |

In [ ]:
class HourlyFlowModule(L.LightningModule):
    def __init__(self, n_features: int, hidden: int, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.net = QuantileLSTM(n_features=n_features, hidden_size=hidden)

    def forward(self, x):
        return self.net(x)

    def _loss(self, batch):
        x, y, *w = batch
        return pinball_loss(self(x), y, self.net._q, w[0] if w else None)

    def training_step(self, batch, batch_idx):
        loss = self._loss(batch)
        self.log("train_pinball", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        self.log("val_pinball", self._loss(batch), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val_pinball"}}


module = HourlyFlowModule(len(H.FEATURES), HIDDEN)

## 7. Training

### The Trainer and its callbacks

| Piece | Role |
|---|---|
| `ModelCheckpoint` | Keeps the weights of the epoch with the lowest `val_pinball` |
| `EarlyStopping` | Stops when `val_pinball` has not improved for 3 epochs |
| `LearningRateMonitor` | Logs the learning rate each epoch, so the scheduler's halvings show in the curves |
| `CSVLogger` | Writes every logged value to `metrics.csv`, read back in section 8 |
| `TensorBoardLogger` | The same for TensorBoard: `tensorboard --logdir data/results/lightning` |
| `gradient_clip_val=1.0` | The clipping caudal's own loop uses |

`accelerator="auto"` uses the Apple GPU (MPS) or CUDA when there is one.

In [ ]:
log_dir = RESULTS_DIR / "lightning"
csv = CSVLogger(log_dir, name=f"hourly_{CONFIG}")
tb = TensorBoardLogger(log_dir, name=f"hourly_{CONFIG}", version=csv.version)
checkpoint = ModelCheckpoint(monitor="val_pinball", mode="min", save_top_k=1, filename="best-{epoch}-{val_pinball:.4f}")
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    gradient_clip_val=1.0,
    callbacks=[checkpoint, EarlyStopping(monitor="val_pinball", patience=3), LearningRateMonitor(logging_interval="epoch")],
    logger=[csv, tb],
    log_every_n_steps=10,
    enable_model_summary=True,
)
t0 = time.time()
trainer.fit(module, train_loader, val_loader)
print(f"trained {trainer.current_epoch} epochs in {time.time() - t0:.0f} s; best checkpoint: {checkpoint.best_model_path}")

## 8. Training curves

The training loss is averaged over each epoch's windows, all leads mixed; the validation loss is at a fixed 24-hour lead on water year 2017. A validation curve that turns up while training keeps falling would be overfitting; early stopping and the checkpoint guard against it.

In [ ]:
m = pd.read_csv(f"{csv.log_dir}/metrics.csv")
per_epoch = m.groupby("epoch").agg(train=("train_pinball_epoch", "last"), val=("val_pinball", "last"))
lr = m["lr-Adam"].dropna().to_numpy()  # Lightning logs the learning rate on rows of its own, one per epoch
per_epoch["lr"] = lr[: len(per_epoch)]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(per_epoch.index, per_epoch["train"], marker="o", label="train, all leads")
ax.plot(per_epoch.index, per_epoch["val"], marker="o", label="validation, 24 h lead")
ax.set_xlabel("epoch"); ax.set_ylabel("weighted pinball loss"); ax.legend()
ax2 = ax.twinx(); ax2.step(per_epoch.index, per_epoch["lr"], where="mid", color="grey", alpha=0.5); ax2.set_ylabel("learning rate")
plt.show()
per_epoch.round(4)

## 9. Scores on the held-out year

### CRPS

For a forecast distribution $F$ and an observation $y$, the continuous ranked probability score is

$$\mathrm{CRPS}(F, y) = \int_{-\infty}^{\infty} \big(F(x) - \mathbf{1}\{x \ge y\}\big)^2\,dx,$$

in cfs, lower is better. It is computed here from the seven quantiles taken as an ensemble (`caudal.scores.crps_ensemble`). Persistence, the newest observed flow carried forward, is the forecast to beat; its CRPS is its absolute error. Skill is $1 - \mathrm{CRPS}_{\text{model}} / \mathrm{CRPS}_{\text{persistence}}$: above 0 the model beats persistence.

For reference, the issued blend, scored the same way on water years 2020 to 2025 with `scripts/compare_hourly.py`, has a CRPS of 23.6 cfs at 1 hour against 31.9 for persistence, and 89.4 at 24 hours against 310.5 (caudal #28). Water year 2017 here is wetter, so its numbers are larger.

In [ ]:
best = HourlyFlowModule.load_from_checkpoint(checkpoint.best_model_path).net.cpu().eval()
rows = []
for lead in (1, 6, 24, 72, 120):
    for lid in GAUGES:
        p = H.predictions(best, frames, lid, lead).dropna(subset=["obs", "last"])
        q = p[[c for c in p.columns if c.startswith("q")]].to_numpy()
        rows.append({"lead": lead, "gauge": lid, "hours": len(p),
                     "model": crps_ensemble(q, p["obs"].to_numpy()).mean(), "persistence": (p["last"] - p["obs"]).abs().mean()})
scores = pd.DataFrame(rows)
table = scores.groupby("lead")[["model", "persistence"]].mean()
table["skill"] = 1 - table["model"] / table["persistence"]
table.round(2)

## 10. A storm: the forecast before the January 2017 flood

The model issues at 00Z on 8 January 2017, knowing the flows up to the hour before, and the weather of the next five days. In this example the weather is what fell (the archive), not the forecast of that day, so it shows the model at its best; the live product reads Open-Meteo's forecast.

In [ ]:
lid, hour = "GUEC1", pd.Timestamp("2017-01-08 00:00")
raw = frames["raw"][lid]
ckpt = {"features": H.FEATURES, "lookback": H.LOOKBACK, "quantiles": list(DEFAULT_QUANTILES), "qmean": frames["qmean"],
        "f_mean": frames["f_mean"], "f_std": frames["f_std"], "scale": frames["scale"]}
from caudal.points import RAIN_POINTS
from caudal.sources import openmeteo, usgs
weather = openmeteo.basin_mean([openmeteo.load(*p) for p in RAIN_POINTS[lid]]).loc[: hour + pd.Timedelta(hours=H.LEADS + 1)]
flows = [usgs.load(s).loc[: hour - pd.Timedelta(hours=1)] for s in H.sites(lid)]
fc = H.issue_point(best, ckpt, lid, hour, weather, flows)
obs = usgs.load(H.sites(lid)[0]).loc[hour - pd.Timedelta(days=2): fc.index[-1]]

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(fc.index, fc["q05"], fc["q95"], alpha=0.2, label="5 to 95%")
ax.fill_between(fc.index, fc["q25"], fc["q75"], alpha=0.35, label="25 to 75%")
ax.plot(fc.index, fc["q50"], lw=2, label="median")
ax.plot(obs.index, obs, color="k", lw=1.5, label="observed")
ax.axvline(hour, color="tab:orange", ls="--", lw=1)
ax.set_ylabel("flow, cfs"); ax.set_title(f"{GAUGES[lid][1]}: issued {hour:%Y-%m-%d %H}Z, leads 1 to {H.LEADS} h"); ax.legend(loc="upper left")
plt.show()
print(f"observed peak {obs.max():,.0f} cfs at {obs.idxmax():%m-%d %H}Z; forecast median peak {fc['q50'].max():,.0f} at {fc['q50'].idxmax():%m-%d %H}Z, "
      f"95% quantile peak {fc['q95'].max():,.0f}")

## Next steps

- Set `FULL_RUN = True` to train at the issued model's size; about 6 minutes on an Apple GPU.
- Set `CONFIG = "full"` to add soil moisture, evaporation and the dam releases, and compare its curves and scores with this run.
- `tensorboard --logdir data/results/lightning` shows every run side by side.
- `scripts/compare_hourly.py` scores trained candidates, HEFS and persistence on the same hours; it is how the issued blend was chosen.